Import the required libraries

In [1]:
import gzip
import io
import os
import re
import time
import zipfile
from pathlib import Path

import requests
import snowflake.connector
from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

Download files from BTS source using through requests

In [ ]:
for year in range(2025, 2027):
    for month in range(1, 13):
        if year == 2026 and month > 7:
            break
        print(f"Downloading data for {year}-{month:02d}...")
        url = f"https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        Path(f"../data/bts/_downloads/ontime_{year}_{month}.zip").write_bytes(r.content)

Create the connection to snowflake and put the credetials in a separate file.
Also created a warehouse in snowflake, a database and a schema to store the data.

In [ ]:
load_dotenv()  # Load environment variables from .env file

conn = snowflake.connector.connect(
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
    schema=os.environ["SNOWFLAKE_SCHEMA"],
)

Unzip the downloaded files locally and push them to the snowflake staging area called @staging_data. 

In [13]:
for file in os.listdir("../data/bts/_downloads/"):
    print(f"Found {file}")
    if file.startswith("ontime_") and file.endswith(".zip"):
        print(f"Processing {file} ...")
        with open(f"../data/bts/_downloads/{file}", "rb") as f:
            zip_data = f.read()
            with zipfile.ZipFile(io.BytesIO(zip_data)) as z:
                for name in z.namelist():
                    if name.endswith(".csv"):
                        # clean output name: no parentheses, ends in .csv
                        out = re.sub(r"[()]", "", os.path.basename(name)) + ".gz"
                        with (
                            z.open(name) as csv_file,
                            gzip.open(out, "wt", encoding="utf-8", newline="") as dst,
                        ):
                            for line in io.TextIOWrapper(
                                csv_file, encoding="utf-8", errors="replace", newline=""
                            ):
                                line = line.rstrip("\r\n")
                                line = line.removesuffix(
                                    ","
                                )  # drop only the one trailing comma
                                dst.write(line + "\n")
                        print(f"Uploading {out}")
                        conn.cursor().execute(
                            f"PUT 'file://{out}' @AVIATION_WEATHER_PROJECT.RAW_DATA.BTS_FLIGHTS AUTO_COMPRESS=FALSE OVERWRITE=TRUE"
                        )

Found .DS_Store
Found ontime_2025_8.zip
Processing ontime_2025_8.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_8.csv.gz
Found ontime_2025_9.zip
Processing ontime_2025_9.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_9.csv.gz
Found ontime_2025_4.zip
Processing ontime_2025_4.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_4.csv.gz
Found T_MASTER_CORD_20261022_091341.zip
Found ontime_2025_5.zip
Processing ontime_2025_5.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_5.csv.gz
Found ontime_2025_7.zip
Processing ontime_2025_7.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_7.csv.gz
Found ontime_2025_6.zip
Processing ontime_2025_6.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_6.csv.gz
Found ontime_2025_2.zip
Processing ontime_2025_2.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Per

So after we downloaded all the files there is an extra step if we are using a trial account. We ideally want to upload this files to snowflake. If not using a trial account, we can directly run this notebook inside snowflake, perhaps it needs some folder creation and changes to code.

After we successfully moved the files to snowflake, now we need to remove them locally.

Lets download carrier lookup data to know exactly what carrier is having most delays.

In [ ]:
url = "https://www.transtats.bts.gov/Download_Lookup.asp?Y11x72=Y_haVdhR_PNeeVRef"
r = requests.get(url, timeout=120)
r.raise_for_status()
Path("../data/bts/carrier_lookup.csv").write_bytes(r.content)

54240

In [18]:
file_name = "../data/bts/carrier_lookup.csv"
print(f"Uploading {file_name}")
conn.cursor().execute(
    f"PUT 'file://{file_name}' @AVIATION_WEATHER_PROJECT.RAW_DATA.BTS_FLIGHTS AUTO_COMPRESS=FALSE OVERWRITE=TRUE"
)

Uploading ../data/bts/carrier_lookup.csv


After downloading the carrier lookup, now we need the airport lookup. So we need to run the download_t_master_cord_selenium.py, we need this lookup data for airport coordinates. I'm adding a separate python execution cause the execution of selenium in this jupyter notebook is crashing python, probably something related to chrome driver. So just run 'python download_airport_cord_selenium.py' and it will download and extract file to the folder

Donwload lookup airport data from BTS using selenium

In [3]:
PAGE_URL = (
    "https://www.transtats.bts.gov/DL_SelectFields.aspx"
    "?gnoyr_VQ=FLL&QO_fu146_anzr=N8vn6v10%20f722146%20gnoyr5"
)
OUT_DIR = Path("../data/bts").resolve()
DOWNLOAD_DIR = OUT_DIR / "_downloads"
FILE_NAME = "AIRPORT_CORD.csv"
OUT_CSV = OUT_DIR / FILE_NAME

HEADLESS = True  # set True to run without opening a browser window
TIMEOUT_SECONDS = 180

# checkboxes that are page options, not table columns
OPTION_KEYWORDS = ("zip", "allgroup", "merge", "null", "doc", "allvar", "selectall")


def make_driver() -> webdriver.Chrome:
    options = webdriver.ChromeOptions()
    if HEADLESS:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,1000")
    options.add_experimental_option(
        "prefs",
        {
            "download.default_directory": str(DOWNLOAD_DIR),
            "download.prompt_for_download": False,
            "download.directory_upgrade": True,
            "safebrowsing.enabled": True,
        },
    )
    return webdriver.Chrome(options=options)


def is_option_box(el) -> bool:
    ident = (
        (el.get_attribute("id") or "") + " " + (el.get_attribute("name") or "")
    ).lower()
    return any(k in ident for k in OPTION_KEYWORDS)


def select_all_fields(driver) -> int:
    """Click 'Select all fields' if it exists, then make sure every column box is checked."""
    boxes = driver.find_elements(By.CSS_SELECTOR, "input[type='checkbox']")

    select_all = [
        b
        for b in boxes
        if any(
            k
            in ((b.get_attribute("id") or "") + (b.get_attribute("name") or "")).lower()
            for k in ("allvar", "selectall")
        )
    ]
    if select_all and not select_all[0].is_selected():
        driver.execute_script("arguments[0].click();", select_all[0])
        time.sleep(1)

    # fallback: tick any column checkbox still unchecked
    checked = 0
    for box in driver.find_elements(By.CSS_SELECTOR, "input[type='checkbox']"):
        if is_option_box(box):
            continue
        if not box.is_selected():
            driver.execute_script("arguments[0].click();", box)
        checked += 1
    return checked


def click_download(driver) -> None:
    xpath = (
        "//button[contains(translate(normalize-space(.), 'DOWNLOAD', 'download'), 'download')]"
        " | //input[(@type='submit' or @type='button') and "
        "contains(translate(@value, 'DOWNLOAD', 'download'), 'download')]"
    )
    button = WebDriverWait(driver, 30).until(
        EC.element_to_be_clickable((By.XPATH, xpath))
    )
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", button)
    driver.execute_script("arguments[0].click();", button)


def wait_for_zip() -> Path:
    deadline = time.time() + TIMEOUT_SECONDS
    while time.time() < deadline:
        zips = list(DOWNLOAD_DIR.glob("*.zip"))
        in_progress = list(DOWNLOAD_DIR.glob("*.crdownload"))
        if zips and not in_progress:
            return max(zips, key=lambda p: p.stat().st_mtime)
        time.sleep(1)
    raise TimeoutError(f"No zip appeared in {DOWNLOAD_DIR} after {TIMEOUT_SECONDS}s")


def extract_csv(zip_path: Path) -> None:
    with zipfile.ZipFile(zip_path) as z:
        csv_name = next(n for n in z.namelist() if n.lower().endswith("cord.csv"))
        OUT_CSV.write_bytes(z.read(csv_name))
    print(f"Saved {FILE_NAME}")


def download_and_extract():
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    for old in DOWNLOAD_DIR.glob("*.zip"):
        old.unlink()

    driver = make_driver()
    try:
        driver.get(PAGE_URL)
        WebDriverWait(driver, 60).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "input[type='checkbox']"))
        )

        n = select_all_fields(driver)
        print(f"Selected {n} field checkboxes")

        click_download(driver)
        print("Download clicked, waiting for the file...")

        extract_csv(wait_for_zip())

    except Exception:
        shot = OUT_DIR / "selenium_error.png"
        driver.save_screenshot(str(shot))
        print(f"Failed screenshot saved to {shot}")
        raise
    finally:
        driver.quit()


download_and_extract()

Selected 33 field checkboxes
Download clicked, waiting for the file...
Saved AIRPORT_CORD.csv


Now Lets send this file to staging_data in snowflake

In [ ]:
conn.cursor().execute(
    "PUT 'file://../data/bts/AIRPORT_CORD.csv' @AVIATION_WEATHER_PROJECT.RAW_DATA.BTS_FLIGHTS AUTO_COMPRESS=FALSE OVERWRITE=TRUE"
)
conn.close()

In [ ]:
# Remove downlaoded files
for file in os.listdir("../data/bts/_downloads/"):
    if file.startswith("ontime_") and file.endswith(("zip","gz")):
        print(f"Removing {file} ...")
        os.remove(f"../data/bts/_downloads/{file}")
# Remove files in the current directory after unzip 
for file in os.listdir():
    if file.endswith((".csv","zip",".gz")):
        print(f"Removing {file} ...")
        os.remove(file)

Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2026_1.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_2.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_11.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2026_3.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_8.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_4.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2026_7.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2026_5.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_6.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_3.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_12.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performan